In [ ]:
import pandas as pd

In [ ]:
# #%%timeit
# with open("/Users/sapirmardan/projects/variant-triage/data/calls_gatk.vcf", "r") as f:
#     l = f.readlines()
#     count = 0
#     for line in l:
#         if line[0:2] == "##":
#             count += 1
#         else:
#             break
#     #print(count)

30 μs ± 8.18 μs per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [20]:
#%%timeit
path_gatk = "../../data/calls_gatk.vcf"
path_bcftools = "../../data/calls.vcf"

with open(path_gatk, "r") as f:
    for i, line in enumerate(f):
        if not line.startswith("##"):
            break
    print(i)

25


In [26]:
import io
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [line for line in f if not line.startswith('##')]
    df = pd.read_csv(
        io.StringIO(''.join(lines)),
        sep='\t',
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str, 'QUAL': str, 'FILTER': str, 'INFO': str, "FORMAT": str, "ecoli": str}
    )
    return df.rename(columns={'#CHROM': 'CHROM'})

gatk_df = read_vcf(path_gatk)
bcf_df = read_vcf(path_bcftools)

gatk_df.head()

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,ecoli
0,CP000819.1,1733343,.,G,A,121.84,.,AC=2;AF=1.00;AN=2;DP=3;ExcessHet=0.0000;FS=0.0...,GT:AD:DP:GQ:PL,"1/1:0,3:3:9:135,9,0"
1,CP000819.1,4504253,.,G,A,64.64,.,AC=1;AF=0.500;AN=2;BaseQRankSum=0.674;DP=4;Exc...,GT:AD:DP:GQ:PL,"0/1:2,2:4:63:72,0,63"


In [27]:
bcf_df.head()

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,data/aligned.sorted.bam
0,CP000819.1,13781,.,C,A,6.51248,.,"DP=1;SGB=-0.379885;MQ0F=0;AC=2;AN=2;DP4=0,0,0,...",GT:PL:AD,"1/1:35,3,0:0,1"
1,CP000819.1,32478,.,G,A,11.7172,.,"DP=1;SGB=-0.379885;MQ0F=0;AC=2;AN=2;DP4=0,0,1,...",GT:PL:AD,"1/1:41,3,0:0,1"
2,CP000819.1,45177,.,C,A,8.99921,.,"DP=1;SGB=-0.379885;MQ0F=0;AC=2;AN=2;DP4=0,0,1,...",GT:PL:AD,"1/1:38,3,0:0,1"
3,CP000819.1,45856,.,C,A,3.77163,.,DP=2;SGB=-0.379885;RPBZ=1;MQBZ=0;BQBZ=0;SCBZ=1...,GT:PL:AD,"0/1:34,0,34:1,1"
4,CP000819.1,89526,.,T,A,10.7923,.,"DP=1;SGB=-0.379885;MQ0F=0;AC=2;AN=2;DP4=0,0,0,...",GT:PL:AD,"1/1:40,3,0:0,1"


In [33]:
left = gatk_df.set_index(["CHROM", "POS"])
right = bcf_df.set_index(["CHROM", "POS"])

gatk_df.join(bcf_df, how="inner", lsuffix="_gatk", rsuffix="_bcf").sort_index(axis=1)


,ALT_bcf,ALT_gatk,CHROM_bcf,CHROM_gatk,FILTER_bcf,FILTER_gatk,FORMAT_bcf,FORMAT_gatk,ID_bcf,ID_gatk,INFO_bcf,INFO_gatk,POS_bcf,POS_gatk,QUAL_bcf,QUAL_gatk,REF_bcf,REF_gatk,data/aligned.sorted.bam,ecoli
0,A,A,CP000819.1,CP000819.1,.,.,GT:PL:AD,GT:AD:DP:GQ:PL,.,.,"DP=1;SGB=-0.379885;MQ0F=0;AC=2;AN=2;DP4=0,0,0,...",AC=2;AF=1.00;AN=2;DP=3;ExcessHet=0.0000;FS=0.0...,13781,1733343,6.51248,121.84,C,G,"1/1:35,3,0:0,1","1/1:0,3:3:9:135,9,0"
1,A,A,CP000819.1,CP000819.1,.,.,GT:PL:AD,GT:AD:DP:GQ:PL,.,.,"DP=1;SGB=-0.379885;MQ0F=0;AC=2;AN=2;DP4=0,0,1,...",AC=1;AF=0.500;AN=2;BaseQRankSum=0.674;DP=4;Exc...,32478,4504253,11.7172,64.64,G,G,"1/1:41,3,0:0,1","0/1:2,2:4:63:72,0,63"
